# 2. Seleção de variáveis: PSI, WoE, IV e correlação

Este notebook mede, para cada variável, a capacidade de separar bons e maus pagadores (WoE e IV), a estabilidade da distribuição ao longo das safras (PSI) e a redundância entre as variáveis (correlação). O resultado é uma ficha com uma linha por variável e o conjunto de variáveis que os modelos dos próximos notebooks usam

A ordem das etapas segue o seguinte funil:
1. Saem as variáveis sem informação (constantes) e as instáveis
2. Depois, saem uma de cada grupo de variáveis redundantes (correlação)
3. Então, os modelos comparam conjuntos de variáveis

Assim a comparação com modelos é feita sobre um conjunto já sem duplicatas.


- O PSI compara cada safra com as duas primeiras (202301 e 202302).
- Os códigos especiais (`-1`, `-2`, `9999`) e os nulos ficam em faixas próprias no WoE e no PSI, porque carregam informação.
- WoE = ln(%bons / %maus): valor positivo indica faixa com mais bons pagadores.
- Testaremos os conjuntos com diferentes remoções sugeridas das variáveis por IV, WOE, PSI e correlação com testes de ganho por um modelo

## 2.1 Imports e leitura da base tratada

In [ ]:
import gc
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from equifax.config import DATA_PROCESSED, TABLES
from equifax.conjuntos import salvar_conjuntos_do_funil
from equifax.experimentos import rodar_experimento
from equifax.selecao import (CODIGOS_ESPECIAIS, correlacao_spearman, ficha_variaveis,
                             pares_correlacionados, psi_por_safra, selecionar_por_correlacao,
                             tabela_woe, woe_por_categoria)
from equifax.validacao import dividir_treino_teste, separar_janelas
from equifax.viz import salvar_figura

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [ ]:
# Base gerada pelo 01_eda (já com CEP formatado e "Sem informação")
df = pd.read_csv(DATA_PROCESSED / "base_des_01.csv", low_memory=False)
df.shape

## 2.2 Janelas e divisão treino/teste

In [ ]:
dev, oot_interna = separar_janelas(df)
treino, teste = dividir_treino_teste(dev)

for nome, parte in [("treino", treino), ("teste", teste), ("oot interna", oot_interna)]:
    print(f"{nome:12} n={len(parte):7} inadimplência={1 - parte['adimplencia'].mean():.4f}")

# A OOT interna não é usada neste notebook
del df, dev, oot_interna
gc.collect();

## 2.3 Lista de variáveis

In [ ]:
VPS = [c for c in treino.columns if c.startswith("VP")]
CATEGORICAS = ["faixa_renda", "faixa_idade", "estado"]
VARIAVEIS = VPS + CATEGORICAS

print(len(VPS), "VPs +", len(CATEGORICAS), "categóricas =", len(VARIAVEIS), "variáveis")

## 2.4 Ficha completa

A Ficha é salva em `reports/tables/`

In [ ]:
ficha, binnings = ficha_variaveis(treino, VARIAVEIS, categoricas=CATEGORICAS)

TABLES.mkdir(parents=True, exist_ok=True)
ficha.to_csv(TABLES / "ficha_variaveis.csv")
print('Ficha das variáveis')
ficha.head(10)

In [ ]:
# Se alguma variável deu erro no binning, ela aparece aqui
ficha[ficha["classe_iv"] == "erro"]

## 2.5 Capacidade preditiva (IV)

Faixas usuais:
* menor que 0,02 sem poder
* 0,02 a 0,10 fraco
* 0,10 a 0,30 médio
* 0,30 a 0,50 forte
* acima de 0,50 suspeito, a revisar

In [ ]:
ordem_iv = ["suspeito (revisar)", "forte", "médio", "fraco", "sem poder", "erro"]
ficha["classe_iv"].value_counts().reindex(ordem_iv, fill_value=0)

In [ ]:
# Distribuição do IV de todas as variáveis, com os limites das classes
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(ficha["iv"], bins=40, color="steelblue")
for limite, rotulo in [(0.02, "0,02"), (0.10, "0,10"), (0.30, "0,30")]:
    ax.axvline(limite, color="gray", linestyle="--")
    ax.text(limite, ax.get_ylim()[1] * 0.95, f" {rotulo}", color="gray", va="top")
ax.set_xlabel("IV")
ax.set_ylabel("Número de variáveis")
ax.set_title("Distribuição do IV das variáveis (treino)")
ax.grid(axis="y", color="gray", alpha=.5)
salvar_figura("selecao_distribuicao_iv")
plt.show()

In [ ]:
# As 20 variáveis com maior IV
colunas = ["iv", "classe_iv", "n_faixas", "psi_max", "classe_psi", "pct_-1", "pct_-2", "pct_9999", "sugestao"]
ficha[colunas].head(20)

In [ ]:
top20 = ficha["iv"].head(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top20.index, top20.values, color="steelblue")
ax.bar_label(ax.containers[0], fmt="%.3f", padding=3)
ax.set_xlim(0, top20.max() * 1.12)
ax.set_xlabel("IV")
ax.set_title("As 20 variáveis com maior IV (treino)")
ax.grid(axis="x", color="gray", alpha=.5)
salvar_figura("selecao_top20_iv")
plt.show()

In [ ]:
# As 5 VPs constantes do diagnóstico devem sair como "remover (constante)"
ficha.loc[ficha["sugestao"] == "remover (constante)", ["n_unicos", "iv", "sugestao"]]

### Variáveis com IV acima de 0,5

In [ ]:
suspeitas = ficha.index[ficha["classe_iv"] == "suspeito (revisar)"].tolist()
print(len(suspeitas), "variáveis com IV > 0,5:", suspeitas)

for var in suspeitas:
    print(f"\n{var} (IV = {ficha.loc[var, 'iv']:.3f})")
    display(tabela_woe(binnings[var]))

### WoE das categóricas e das 5 melhores VPs

In [ ]:
melhores = [v for v in ficha.index if v in VPS and v not in suspeitas][:5]

for var in CATEGORICAS + melhores:
    print(f"\n{var} (IV = {ficha.loc[var, 'iv']:.3f})")
    display(tabela_woe(binnings[var]))

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

### WoE de cada categoria, sem agrupar

O binning ótimo junta categorias com comportamento semelhante. A tabela mostra cada categoria separada, para ver quais vizinhas tem WoE próximo e poderiam ser fundidas

In [ ]:
def ordem_natural(rotulo):
    """Ordena as faixas pelo primeiro número do rótulo; 'Sem informação' vai para o fim."""
    numero = re.match(r"\s*(\d+)", str(rotulo))
    return int(numero.group(1)) if numero else float("inf")


tabelas_categoria = {}
for var in CATEGORICAS:
    tabela = woe_por_categoria(treino[var], treino["adimplencia"])
    if var == "estado":
        tabela = tabela.sort_values("WoE")
    else:
        tabela = tabela.loc[sorted(tabela.index, key=ordem_natural)]
    tabelas_categoria[var] = tabela
    print(f"\n{var}: IV sem agrupar = {tabela['IV'].sum():.4f}; IV com binning ótimo = {ficha.loc[var, 'iv']:.4f}")
    display(tabela.round(4))

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

## 2.6 Estabilidade (PSI)

PSI de cada safra do treino contra 202301 e 202302. Abaixo de 0,10 estável; 0,10 a 0,25 atenção; acima de 0,25 instável. A classificação usa o maior PSI entre as safras.

In [ ]:
ficha["classe_psi"].value_counts().reindex(["estável", "atenção", "instável", "sem dado"], fill_value=0)

In [ ]:
# PSI safra a safra das variáveis em atenção ou instáveis
fora = ficha.index[ficha["classe_psi"].isin(["atenção", "instável"])].tolist()
print(len(fora), "variáveis em atenção ou instáveis")

if fora:
    psi_safras = pd.DataFrame({var: psi_por_safra(treino, var) for var in fora}).T
    psi_safras["iv"] = ficha.loc[fora, "iv"]
    display(psi_safras.sort_values(psi_safras.columns[-2], ascending=False).round(3))

In [ ]:
# Cruzamento das duas classificações
pd.crosstab(ficha["classe_iv"], ficha["classe_psi"]).reindex(ordem_iv).dropna(how="all").astype(int)

In [ ]:
# IV x PSI: cada ponto é uma variável. As melhores ficam à direita e embaixo.
instaveis = ficha.index[ficha["classe_psi"] == "instável"].tolist()

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(ficha["iv"], ficha["psi_max"].clip(lower=1e-4), s=18, color="steelblue", alpha=.7)
for var in instaveis:
    ax.annotate(var, (ficha.loc[var, "iv"], ficha.loc[var, "psi_max"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.axhline(0.10, color="gray", linestyle="--")
ax.axhline(0.25, color="gray", linestyle="--")
ax.axvline(0.02, color="gray", linestyle=":")
ax.set_yscale("log")
ax.set_xlabel("IV")
ax.set_ylabel("PSI máximo entre as safras (escala log)")
ax.set_title("Capacidade preditiva x estabilidade (treino)")
ax.grid(color="gray", alpha=.3)
salvar_figura("selecao_iv_x_psi")
plt.show()

### Interpretação das variáveis instáveis

O gráfico mostra em que safra o PSI de cada uma sobe. A tabela seguinte resume, para cada variável, a safra da quebra comparada com as outras safras do treino: a participação do zero e a mediana dos valores normais (sem códigos especiais).

In [ ]:
if instaveis:
    fig, ax = plt.subplots(figsize=(9, 5))
    for var in instaveis:
        ax.plot(psi_safras.columns[:-1].astype(str), psi_safras.loc[var].iloc[:-1].astype(float).clip(lower=1e-4), marker="o", label=var)
    ax.axhline(0.25, color="gray", linestyle="--")
    ax.set_yscale("log")
    ax.set_xlabel("Safra")
    ax.set_ylabel("PSI contra 202301-202302 (escala log)")
    ax.set_title("PSI por safra das variáveis instáveis (treino)")
    ax.legend()
    ax.grid(color="gray", alpha=.3)
    salvar_figura("selecao_psi_instaveis")
    plt.show()

In [ ]:
def resumo_da_quebra(var):
    """Compara a safra em que o PSI salta com as demais safras do treino (só agregados)."""
    psi_var = psi_safras.loc[var].drop("iv").astype(float)
    safra_quebra = psi_var.idxmax()
    x = treino[var]
    na_quebra = treino["safra"] == safra_quebra
    normais = x.where(~x.isin(CODIGOS_ESPECIAIS))
    return {
        "safra da quebra": safra_quebra,
        "PSI na safra": psi_var.max(),
        "% de zeros na safra": (x[na_quebra] == 0).mean(),
        "% de zeros nas outras": (x[~na_quebra] == 0).mean(),
        "mediana na safra": normais[na_quebra].median(),
        "mediana nas outras": normais[~na_quebra].median(),
        "IV": ficha.loc[var, "iv"],
    }


quebras = pd.DataFrame({var: resumo_da_quebra(var) for var in instaveis if var in VPS}).T
quebras["safra da quebra"] = quebras["safra da quebra"].astype(int)
quebras.round(3)

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

## 2.7 Correlação entre as variáveis

Duas variáveis muito correlacionadas carregam quase a mesma informação. Por isso, manter as duas não ajuda no modelo, apenas dificulta a interpretação e distorce a medida de importância de cada uma. Então, faremos a análise de correlação antes dos testes com modelo:

- Tiramos as variáveis constantes e as instáveis. As três categóricas (`faixa_renda`, `faixa_idade` e `estado`) também ficam de fora, pois correlação só se aplica a variáveis numéricas.
- Considerando os códigos especiais que encontramos na EDA (`9999`, `-1`, etc.), optamos por usar a correlação de **Spearman**, pois ela compara a ordem dos valores e não é distorcida por esses valores. Assim, por exemplo, `9999` conta só como o maior valor, não como um número enorme, como aconteceria na correlação de Pearson
- As variáveis são percorridas do maior IV para o menor. Cada uma fica na seleção se não tiver correlação em módulo igual ao limiar (que testaremos e escolheremos mais a frente) ou mais com alguma que já ficou. Dessa maneira, em cada grupo de variáveis parecidas fica a de maior IV
- Calculada só no treino

In [ ]:
constantes = ficha.index[ficha["sugestao"] == "remover (constante)"].tolist()
VPS_CORRELACAO = [v for v in VPS if v not in constantes + instaveis]
LIMIAR_CORRELACAO = 0.90

corr = correlacao_spearman(treino, VPS_CORRELACAO)
print(len(VPS_CORRELACAO), "VPs na análise de correlação")

### Mapa de calor da correlação

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="RdBu_r", vmin=-1, vmax=1, xticklabels=20, yticklabels=20,
            cbar_kws={"label": "Correlação de Spearman"}, ax=ax)
ax.set_title(f"Correlação de Spearman entre as {len(VPS_CORRELACAO)} VPs (treino)")
plt.tight_layout()
salvar_figura("selecao_correlacao")
plt.show()

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

### Escolha do limiar da correlação

In [ ]:
# Quantos pares e quantas variáveis sairiam com cada limiar (a proposta usa 0,90)
sensibilidade = []
for limiar in [0.80, 0.85, 0.90, 0.95, 0.99]:
    _, saem_limiar = selecionar_por_correlacao(corr, ficha["iv"], limiar)
    sensibilidade.append({"limiar": limiar, "pares acima do limiar": len(pares_correlacionados(corr, limiar)),
                          "variáveis que saem": len(saem_limiar),
                          "VPs que ficam": len(VPS_CORRELACAO) - len(saem_limiar)})
pd.DataFrame(sensibilidade)

Escolhemos o limiar de **0,90**, que é o valor já definido no código acima (`LIMIAR_CORRELACAO`). Acima dele, as duas variáveis do par trazem quase a mesma informação. Com limiares mais baixos, sairiam variáveis que ainda podem ter informação própria, e modelos de árvore como o LightGBM lidam bem com correlação moderada entre as variáveis. O custo desse corte no modelo é medido na seção 2.8

Na regressão logística do Modelo Base (notebook 03), que é mais sensível a variáveis correlacionadas, usamos um limiar mais baixo: 0,70 ou 0,80, escolhido na validação cruzada e aplicado à correlação entre os WoE

### Variáveis que saem com o limiar de 0,90

As três tabelas abaixo mostram os pares mais correlacionados, a lista das variáveis que saem (com a variável parecida que fica no lugar) e quais delas estão entre as 20 de maior IV

In [ ]:
# Os pares mais correlacionados, com o IV de cada variável
pares = pares_correlacionados(corr, LIMIAR_CORRELACAO)
pares["iv_1"] = ficha.loc[pares["var_1"], "iv"].values
pares["iv_2"] = ficha.loc[pares["var_2"], "iv"].values
print(len(pares), f"pares com correlação de {LIMIAR_CORRELACAO:.2f} ou mais")
pares.head(20).round(4)

In [ ]:
# Qual variável sai e qual variável fica no lugar
ficam_corr, saem_corr = selecionar_por_correlacao(corr, ficha["iv"], LIMIAR_CORRELACAO)

saidas_corr = pd.DataFrame({"fica_no_lugar": pd.Series(saem_corr)})
saidas_corr["correlacao"] = [corr.loc[sai, fica] for sai, fica in saem_corr.items()]
saidas_corr["iv"] = ficha.loc[saidas_corr.index, "iv"]
saidas_corr["iv_de_quem_fica"] = ficha.loc[saidas_corr["fica_no_lugar"], "iv"].values
saidas_corr["classe_iv"] = ficha.loc[saidas_corr.index, "classe_iv"]
saidas_corr = saidas_corr.sort_values("iv", ascending=False)
saidas_corr.to_csv(TABLES / "saidas_correlacao.csv")

print(len(saidas_corr), "variáveis saem por correlação;", len(ficam_corr), "VPs ficam")
display(saidas_corr["classe_iv"].value_counts())
saidas_corr.round(4)

In [ ]:
# Variáveis do top 20 por IV que saem, e quem fica no lugar delas
saidas_corr.loc[[v for v in ficha.index[:20] if v in saem_corr], ["fica_no_lugar", "correlacao", "iv", "iv_de_quem_fica"]].round(4)

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

## 2.8 Teste com e sem as candidatas a remoção

Por enquanto, cada corte foi decidido olhando para as variáveis sozinhas (constantes e PSI) ou em pares (correlação). Agora testaremos com um modelo se esses cortes custam desempenho, e decidiremos o que fazer com as variáveis de IV abaixo de 0,02, avaliando o ganho:

- Usamos um LightGBM com parâmetros fixos, sem ajuste, pois ele serve só para comparar os conjuntos. O ajuste do modelo fica para o notebook 03
- Cada conjunto é treinado no treino e avaliado no teste, com 3 sementes. Assim, conseguimos separar o efeito do corte da variação entre execuções
- O treino e a avaliação vem do script de experimentos (`src/equifax/experimentos.py`), e cada execução fica registrada em `reports/experimentos.csv`

Conjuntos comparados:

1. **Com todas:** todas as variáveis
2. **Sem constantes e instáveis:** saem as constantes e as instáveis
3. **Sem correlacionadas:** saem também as VPs redundantes da análise de correlação
4. **Sem correlacionadas e sem IV < 0,02:** saem também todas as variáveis de IV abaixo de 0,02
5. **Proposta:** como o 3, mas são removidas apenas as variáveis de IV abaixo de 0,02 que o modelo menos usa (a descrição da regra está adiante)

In [ ]:
PARTES = {"treino": treino, "teste": teste}  # a OOT interna não entra
RESULTADOS, GANHOS = [], {}


def rodar_conjunto(nome, variaveis):
    """Roda um conjunto de variáveis com as 3 sementes e guarda métricas e ganho médio.

    Usa o motor de experimentos: LightGBM de referência (os mesmos parâmetros de
    antes), treino no treino, avaliação no teste com `avaliar`. Cada semente vira
    uma linha em reports/experimentos.csv. O ganho é a parte da redução do erro,
    no treino, que veio dos cortes feitos em cada variável (soma 1 entre todas).
    """
    res = rodar_experimento("02_selecao_com_e_sem", list(variaveis), PARTES, modelo="lgbm",
                            nome_conjunto=nome)
    metricas = res.por_safra[["semente", "safra", "n", "taxa_inad", "auc", "gini", "ks"]].copy()
    metricas.insert(0, "n_variaveis", len(variaveis))
    metricas.insert(0, "conjunto", nome)
    RESULTADOS.append(metricas)
    GANHOS[nome] = res.importancia
    print(f"{nome}: {len(variaveis)} variáveis, {len(res.registro)} sementes")


def comparar():
    """Média e desvio entre as sementes, no teste inteiro, de cada conjunto já rodado."""
    resultados = pd.concat(RESULTADOS, ignore_index=True)
    total = resultados[resultados["safra"] == "todas"]
    return (total.groupby(["conjunto", "n_variaveis"])[["auc", "gini", "ks"]]
            .agg(["mean", "std"]).round(4).sort_index(level="n_variaveis", ascending=False))


def ks_por_safra():
    """KS médio das sementes em cada safra do teste, de cada conjunto já rodado."""
    resultados = pd.concat(RESULTADOS, ignore_index=True)
    return (resultados[resultados["safra"] != "todas"]
            .groupby(["safra", "conjunto"])["ks"].mean().unstack().round(4))

#### Rodando modelos 1, 2 e 3


In [ ]:
com_todas = VARIAVEIS
sem_instaveis = [v for v in VARIAVEIS if v not in constantes + instaveis]
sem_correlacionadas = [v for v in sem_instaveis if v not in saem_corr]

rodar_conjunto("com todas", com_todas)
rodar_conjunto("sem constantes e instáveis", sem_instaveis)
rodar_conjunto("sem correlacionadas", sem_correlacionadas)

### Candidatas a remoção que o modelo mais usa

**IV**: mede cada variável sozinha \
**Ganho**: mede quanto o modelo usou cada uma em combinação com as outras

Regra testada: entre as variáveis que sobraram da correlação, sai a de IV abaixo de 0,02 cujo ganho fica abaixo da mediana do ganho de todas as variáveis do modelo sem correlacionadas. As que o modelo usa mais do que a metade das variáveis ficam

Usamos o ganho proveniente do modelo sem correlacionadas pois, quando duas variáveis trazem a mesma informação, o modelo usa uma hora uma, outra hora outra, e o ganho se divide entre ambas. Assim, o ganho de cada uma não refletiria sua real influência

A mediana foi escolhida para ter um corte simples de testar. É a mesma regra da função `SelectFromModel(threshold="median")` do scikit-learn, aplicada só para as variáveis de IV abaixo de 0,02

In [ ]:
ganho = GANHOS["sem correlacionadas"]
ficha["ganho"] = ganho  # fica vazio para as variáveis que saíram antes
ficha["posicao_ganho"] = ganho.rank(ascending=False)

candidatas = [v for v in sem_correlacionadas if ficha.loc[v, "sugestao"] == "candidata a remoção (testar)"]
mediana_ganho = ganho.median()
pouco_usadas = [v for v in candidatas if ganho[v] < mediana_ganho]
usadas = [v for v in candidatas if v not in pouco_usadas]

print(f"{len(candidatas)} candidatas (IV < 0,02) somam {ganho[candidatas].sum():.1%} do ganho do modelo")
print(len(pouco_usadas), "com ganho abaixo da mediana;", len(usadas), "acima")
ficha.to_csv(TABLES / "ficha_variaveis.csv")

# Candidatas que o modelo mais usa, apesar do IV baixo
ficha.loc[usadas, ["iv", "psi_max", "classe_psi", "ganho", "posicao_ganho"]].sort_values("posicao_ganho").head(15)

In [ ]:
# As 20 variáveis com maior ganho, para comparar com o top 20 por IV
ficha.loc[sem_correlacionadas].sort_values("posicao_ganho")[["iv", "classe_iv", "ganho", "posicao_ganho"]].head(20)

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

In [ ]:
sem_iv_baixo = [v for v in sem_correlacionadas if v not in candidatas]
proposta = [v for v in sem_correlacionadas if v not in pouco_usadas]

rodar_conjunto("sem correlacionadas e sem IV < 0,02", sem_iv_baixo)
rodar_conjunto("proposta", proposta)

### Comparação dos conjuntos

Média e desvio das 3 sementes no teste inteiro, e depois o KS de cada safra do teste

Se a diferença entre dois conjuntos for menor que o desvio entre as sementes, o corte não muda o resultado. Se o modelo com menos variáveis piorar, o corte não deve ser feito inteiro

In [ ]:
resultados = pd.concat(RESULTADOS, ignore_index=True)
resultados.to_csv(TABLES / "teste_com_e_sem.csv", index=False)

comparacao = comparar()
comparacao

In [ ]:
# KS por safra
ks_safra = ks_por_safra()
ks_safra

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.

## 2.9 Funil final

A tabela abaixo resume quantas variáveis sobram depois de cada etapa.

In [ ]:
funil = pd.DataFrame({
    "etapa": ["Variáveis iniciais", "Sem as constantes", "Sem as instáveis no PSI",
              "Sem as correlacionadas (uma por grupo, limiar 0,90)",
              "Sem as de IV < 0,02 pouco usadas pelo modelo (proposta)"],
    "n_variaveis": [len(VARIAVEIS), len(VARIAVEIS) - len(constantes), len(sem_instaveis),
                    len(sem_correlacionadas), len(proposta)],
})
funil

## 2.10 Conjuntos versionados

Os conjuntos do funil são gravados em `config/conjuntos.yaml`, que guarda só os nomes das colunas. Os modelos dos próximos notebooks leem as listas de lá, então trocar de conjunto exige só uma nova execução, sem mudar código

- Aqui são gravados quatro conjuntos: `todas`, `sem_const_inst`, `sem_corr_090` e o conjunto proposto (`proposta_<n>`, em que `n` é o número de variáveis)
- Todas as variáveis desses conjuntos são colunas fornecidas, como o regulamento pede para o Modelo Base. Por isso o notebook 03 usa o conjunto proposto direto
- Se um conjunto já existir com outra lista (por exemplo, depois de mudar o limiar de correlação), a função para com erro. Para trocar de propósito, é preciso passar `sobrescrever=True`

In [ ]:
salvar_conjuntos_do_funil(ficha, saidas_corr)